In [ ]:
#pip install transformers torch

In [ ]:
# Loading libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Loading model
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
model.to("mps")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

In [ ]:
# Function to generate chatbot response
def generate_response(user_input, messages, max_history=10):

    messages.append({"role":"user","content":user_input})
    if len(messages) > max_history:
        messages = messages[-max_history:]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("mps")
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1
    )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:],skip_special_tokens=True)
    messages.append({"role":"assistant","content":response})

    return response, messages

In [ ]:
# Function to run the chatbot till quit
def run_chatbot():

    print("-"*35)
    print("Hi! I am your Chat Assistant!")
    print("Type 'exit' or 'quit' to end")
    print("-"*35)

    messages = [
        {
        "role":"system",
        "content":"You are a helpful AI assistant. Give clear and correct answers."
        }
    ]

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ["exit","quit"]:
            print("Thank you for your time!")
            print("-" * 35)
            break
        if not user_input:
            print("Chatbot: Please submit a response!")
            continue

        response, messages = generate_response(user_input, messages)
        if not response.strip():
            response = "Could you rephrase that?"

        print("Chatbot:",response)

In [ ]:
run_chatbot()

-----------------------------------
Hi! I am your Chat Assistant!
Type 'exit' or 'quit' to end
-----------------------------------


You:  Hi


Chatbot: Hello! How can I help you today?


You:  What is Artificial Intelligence?


Chatbot: Artificial intelligence, often abbreviated as AI, refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions. The term may also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving. This technology has evolved from simple algorithms designed for specific tasks into complex systems capable of performing multiple functions across various domains including language translation, image recognition, autonomous vehicles, and personalized recommendations on streaming services or shopping platforms.


You:  Describe about python in a few short sentences


Chatbot: Python is an interpreted high-level programming language known for its simplicity and readability which makes it ideal for beginners but powerful enough for professional developers. It supports multiple programming paradigms, including object-oriented, imperative, functional, procedural, and aspect-oriented styles. With extensive standard libraries supporting many common programming tasks quickly and easily, along with third-party modules, Python enables rapid development without compromising quality or scalability. Its widespread use spans web development (Django framework), data analysis (Pandas library), artificial intelligence (TensorFlow), scientific computing (SciPy stack) among others.


You:  Who created Python programming language?


Chatbot: Guido van Rossum began work on what would become Python during 1989 while working at Centrum Wiskunde & Informatica (CWI) in the Netherlands. He was inspired by several languages he had used before - ABC, Modula-3, and C++ – aiming to create something more accessible yet still robust. Van Rossum released his first version under the name "Benevolent Dictator For Life" (BDFL). After stepping down from this role voluntarily in July 2018 due to health reasons, Alan Turing Award winner Tim Peters took over responsibility for ensuring consistent improvements within the core software base; however, Jeffrey Yasskin then became BDF
